# 18.5 Testing API Clients and Concurrency

**Prerequisites:** 18.1–18.4, 15.5 Test Doubles, 12.4 concurrent.futures, 12.5 asyncio  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 **Do not mock `requests`** — wrap the API and fake your own boundary (**15.5**)
- Three layers of test: unit with a fake, integration against a local server, contract
- Why an I/O-bound program shows nothing useful in a profiler (**17.5**)
- **Threads** for concurrent HTTP — `ThreadPoolExecutor` (**12.4**), measured
- **asyncio** with `httpx` (**12.5**) — and 🔴 an honest measurement of when it loses
- 🔴 **Connection pooling**, measured: the cheapest win there is
- Bounded concurrency, because a rate limit does not care how fast you are (**18.3**)
- Interview questions

---

## Two questions this folder has not answered

You can now talk to an API (**18.1**), authenticate (**18.2**), paginate and retry (**18.3**)
and validate what comes back (**18.4**). Two things remain:

1. **How do you test any of it** without hitting the real service?
2. **Why is it so slow**, and what actually helps?

They are connected: both are answered by putting a **boundary** around the API.

In [ ]:
# ---- A fake API with a controllable delay ----
import http.server
import json
import socket
import threading
import time
import urllib.parse

DELAY = {"seconds": 0.15}          # pretend network latency; changed per experiment
REQUEST_COUNT = {"n": 0}


class LatentAPI(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        """Silence the default logging."""

    def do_GET(self):
        REQUEST_COUNT["n"] += 1
        if DELAY["seconds"]:
            time.sleep(DELAY["seconds"])
        path = urllib.parse.urlparse(self.path).path
        if path == "/boom":
            payload, status = {"error": "internal"}, 500
        else:
            payload, status = {"id": path.rsplit("/", 1)[-1], "state": "done"}, 200
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


class QuietServer(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, *args):
        """A client hanging up is normal."""


def start_api():
    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    server = QuietServer(("127.0.0.1", port), LatentAPI)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{port}"


SERVER, BASE = start_api()
print("fake API on", BASE, f"| artificial latency {DELAY['seconds'] * 1000:.0f} ms/request")

## The client under test

Everything below tests this class. Note what it does *not* do: it never leaks `requests` into
the rest of the program — the same "wrap what you do not own" argument **15.5** made for
mocking and **16.5** made for untyped libraries.

In [ ]:
from dataclasses import dataclass

import requests


@dataclass(frozen=True)
class Job:
    id: str
    state: str

    @property
    def is_finished(self) -> bool:
        return self.state in {"done", "failed"}


class JobsClient:
    """The only class in the program that knows the API exists."""

    def __init__(self, base_url, session=None, timeout=5.0):
        self._base = base_url.rstrip("/")
        self._session = session or requests.Session()
        self._timeout = timeout

    def get_job(self, job_id: str) -> Job:
        response = self._session.get(f"{self._base}/jobs/{job_id}",
                                     timeout=self._timeout)
        response.raise_for_status()
        payload = response.json()
        return Job(id=payload["id"], state=payload["state"])

    def close(self):
        self._session.close()


client = JobsClient(BASE)
job = client.get_job("build-001")
print("fetched:", job, "| finished:", job.is_finished)

## 🔴 Do not mock `requests`

The tempting test is `patch("requests.get")`. **15.5** explained why it is wrong, and it applies
with full force here:

> Mocking `requests` encodes **your belief** about how `requests` behaves. If that belief is
> wrong — or the library changes — your tests stay green and production breaks.

There is a second problem specific to HTTP: a `Mock` response has no status code semantics, no
header parsing, no `raise_for_status`. You end up reimplementing `requests` badly inside your
test fixtures.

**Fake at your own boundary instead.** Two good options:

| Approach | Fake what | Good for |
|---|---|---|
| **A fake transport** | the `Session` object your client accepts | unit tests, fast, no sockets |
| **A local server** | the API itself | 🔴 integration tests — exercises real HTTP |
| A library (`responses`, `respx`) | `requests`/`httpx` internals | convenient, but couples you to the library |

The client above takes `session=` precisely so a test can pass something else — dependency
injection (**15.5**) rather than patching.

In [ ]:
import textwrap
import subprocess
import sys
import tempfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py185_"))


def write(rel, source):
    path = WORK / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


# The client, as an importable module.
write("jobs_client.py", r"""
    from dataclasses import dataclass

    import requests


    @dataclass(frozen=True)
    class Job:
        id: str
        state: str

        @property
        def is_finished(self) -> bool:
            return self.state in {"done", "failed"}


    class JobsClient:
        def __init__(self, base_url, session=None, timeout=5.0):
            self._base = base_url.rstrip("/")
            self._session = session or requests.Session()
            self._timeout = timeout

        def get_job(self, job_id):
            response = self._session.get(f"{self._base}/jobs/{job_id}",
                                         timeout=self._timeout)
            response.raise_for_status()
            payload = response.json()
            return Job(id=payload["id"], state=payload["state"])
""")

write("test_unit.py", r"""
    import pytest
    import requests

    from jobs_client import Job, JobsClient


    class FakeSession:
        # A fake at OUR boundary: it answers .get(), nothing more.

        def __init__(self, responses):
            self._responses = responses
            self.calls = []

        def get(self, url, timeout=None):
            self.calls.append((url, timeout))
            return self._responses.pop(0)


    class FakeResponse:
        def __init__(self, status, payload):
            self.status_code = status
            self._payload = payload

        def json(self):
            return self._payload

        def raise_for_status(self):
            if self.status_code >= 400:
                raise requests.HTTPError(f"{self.status_code}", response=self)


    def test_parses_a_job():
        session = FakeSession([FakeResponse(200, {"id": "b-1", "state": "done"})])
        client = JobsClient("http://api.test", session=session)

        job = client.get_job("b-1")

        assert job == Job(id="b-1", state="done")
        assert job.is_finished is True


    def test_sends_a_timeout():
        # 🔴 A test that a real mock of `requests` would not make obvious.
        session = FakeSession([FakeResponse(200, {"id": "b-1", "state": "queued"})])
        JobsClient("http://api.test", session=session, timeout=2.5).get_job("b-1")

        url, timeout = session.calls[0]
        assert url == "http://api.test/jobs/b-1"
        assert timeout == 2.5


    def test_raises_on_500():
        session = FakeSession([FakeResponse(500, {"error": "internal"})])
        client = JobsClient("http://api.test", session=session)

        with pytest.raises(requests.HTTPError):
            client.get_job("b-1")
""")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header",
     "test_unit.py"],
    cwd=WORK, capture_output=True, text=True, encoding="utf-8", timeout=300)
print("$ pytest test_unit.py")
print("-" * 68)
print((result.stdout + result.stderr).strip())

Three tests, no sockets, milliseconds to run. Note the second one: it asserts
that the client **passed the timeout through** — the kind of check that matters (**11.5**: a
request without a timeout can hang forever) and that a `patch("requests.get")` makes awkward.

🔴 **But these tests cannot catch everything.** They prove your *parsing* is right. They cannot
prove the API's URL shape is right, that a real `500` looks like you think, or that a real
`raise_for_status` behaves as your fake does. For that you need the next layer.

## Integration tests: a real server

Run the API for real — locally — and exercise the whole stack including sockets, headers and
status codes. This is what the whole folder has been doing.

In [ ]:
write("conftest.py", r"""
    import http.server
    import json
    import socket
    import threading

    import pytest


    class TinyAPI(http.server.BaseHTTPRequestHandler):
        protocol_version = "HTTP/1.1"

        def log_message(self, *args):
            pass

        def do_GET(self):
            if self.path.endswith("/missing"):
                payload, status = {"error": "not found"}, 404
            else:
                payload, status = {"id": self.path.rsplit("/", 1)[-1],
                                   "state": "done"}, 200
            body = json.dumps(payload).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)


    class QuietServer(http.server.ThreadingHTTPServer):
        daemon_threads = True

        def handle_error(self, *args):
            pass


    @pytest.fixture(scope="session")
    def api_url():
        # A real HTTP server for the whole test session (15.4).
        probe = socket.socket()
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
        probe.close()
        server = QuietServer(("127.0.0.1", port), TinyAPI)
        thread = threading.Thread(target=server.serve_forever, daemon=True)
        thread.start()
        yield f"http://127.0.0.1:{port}"
        server.shutdown()
""")

write("test_integration.py", r"""
    import pytest
    import requests

    from jobs_client import JobsClient


    @pytest.fixture
    def client(api_url):
        client = JobsClient(api_url)
        yield client
        client._session.close()


    def test_fetches_over_real_http(client):
        job = client.get_job("build-001")
        assert job.id == "build-001"
        assert job.is_finished is True


    def test_404_becomes_an_exception(client):
        with pytest.raises(requests.HTTPError) as caught:
            client.get_job("missing")
        assert caught.value.response.status_code == 404
""")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--no-header",
     "test_unit.py", "test_integration.py"],
    cwd=WORK, capture_output=True, text=True, encoding="utf-8", timeout=300)
print("$ pytest test_unit.py test_integration.py")
print("-" * 68)
print((result.stdout + result.stderr).strip())

Five tests: three fast unit tests and two that speak real HTTP over a real
socket — and the whole suite still runs offline.

🔴 **The session-scoped fixture (**15.4**) starts the server once for the entire run**, not per
test. A per-test server would work and would be slower for no benefit.

### The three layers

| Layer | Fakes | Catches | Speed |
|---|---|---|---|
| **Unit** | your `Session` boundary | parsing, error mapping, timeouts | µs |
| **Integration** | nothing (local server) | 🔴 URL shapes, real status handling, headers | ms |
| **Contract** | nothing (the *real* API, occasionally) | 🔴 **the provider changing** | seconds, and flaky |

Mark the third `@pytest.mark.integration` (**15.3**) and exclude it from the normal run:
`pytest -m "not integration"`. You want it to exist, and you do not want it in your pre-commit
hook.

> **Recorded fixtures** (`vcrpy`, `betamax`) record real responses once and replay them. Useful,
> with one hazard worth naming: 🔴 a recording **freezes the API as it was on the day you
> recorded**, so your tests keep passing after the provider changes. They also frequently capture
> credentials into the recording file — check before committing (**18.2**).

## 🔴 Why it is slow: the profiler shows nothing

**17.5** ended with the warning that a 95% I/O-bound program shows almost nothing useful in
`cProfile`. This is that program.

In [ ]:
import cProfile
import io
import pstats

session = requests.Session()
client = JobsClient(BASE, session=session)
PATHS = [f"build-{i:03d}" for i in range(10)]

profiler = cProfile.Profile()
profiler.enable()
started = time.perf_counter()
for job_id in PATHS:
    client.get_job(job_id)
elapsed = time.perf_counter() - started
profiler.disable()

buffer = io.StringIO()
pstats.Stats(profiler, stream=buffer).strip_dirs().sort_stats("tottime").print_stats(5)
print(buffer.getvalue())
print(f"wall clock: {elapsed * 1000:.0f} ms for {len(PATHS)} requests")
print()
print("🔴 Read the tottime column. Almost nothing. The program spent its life")
print("   BLOCKED in a socket read, which is not CPU time and does not appear.")
print("   No amount of code optimisation (17.5) will help. The answer is to")
print("   stop waiting one request at a time.")

The profile is essentially empty while the wall clock says seconds. That is
the signature of I/O-bound work, and it is why **12** exists.

## Threads: `ThreadPoolExecutor`

**12.1** established that the GIL prevents threads from speeding up CPU-bound work. **12.4**
established the exception: **threads are ideal for I/O-bound work**, because a thread waiting
on a socket has released the GIL.

Ten requests that each take 150 ms are not 1.5 seconds of work — they are 1.5 seconds of
*waiting*, and waiting parallelises perfectly.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

TARGETS = [f"build-{i:03d}" for i in range(20)]


def fetch_serial():
    with requests.Session() as ses:
        api = JobsClient(BASE, session=ses)
        return [api.get_job(j) for j in TARGETS]


def fetch_threaded(workers):
    with requests.Session() as ses:
        api = JobsClient(BASE, session=ses)
        with ThreadPoolExecutor(max_workers=workers) as pool:
            return list(pool.map(api.get_job, TARGETS))


timings = {}
for label, call in (("serial", fetch_serial),
                    ("threads (5)", lambda: fetch_threaded(5)),
                    ("threads (10)", lambda: fetch_threaded(10)),
                    ("threads (20)", lambda: fetch_threaded(20))):
    started = time.perf_counter()
    jobs = call()
    timings[label] = time.perf_counter() - started
    print(f"   {label:14} {len(jobs):3} jobs in {timings[label] * 1000:7.0f} ms")

baseline = timings["serial"]
print()
for label, elapsed in timings.items():
    print(f"   {label:14} {baseline / elapsed:5.1f}x vs serial")
best = min(timings, key=timings.get)
print()
print(f"🔴 20 requests x 150 ms of waiting = 3 s serially. Best here: {best}.")
print("   Note that MORE workers is not monotonically better - see the note below.")

The shape is clear: **20 requests of 150 ms each is 3 seconds of pure
waiting**, and overlapping that waiting is worth roughly an order of magnitude.

🔴 **Now run the cell again.** The ordering of the last two rows is **not stable** on this
setup — sometimes 20 workers is the fastest, sometimes 10 beats it comfortably. That is not a
flaw in the demonstration; it is the point:

1. **Most of the win arrives early.** Going from serial to ~5–10 workers is dramatic. Beyond
   that you are competing for a resource that is no longer yours.
2. **The bottleneck moves to the other side.** This `ThreadingHTTPServer` spawns an **OS thread
   per connection**, so at high concurrency the *server* becomes the constraint — and a
   contended server gives different numbers every run.

> **The lesson is not a number of workers.** It is that concurrency has an optimum, the optimum
> is set by **the other side**, and you find it by measuring your own workload (**17.5**) —
> exactly as **14.1** concluded when wall-clock timing proved too noisy to teach from. A real
> API also tells you its limit, as a rate limit (**18.3**).

## asyncio with `httpx`, and 🔴 an honest measurement

**12.5** built `asyncio`. The conventional wisdom is that async is the right tool for many
concurrent network calls. Let us measure it rather than assert it.

`requests` is synchronous and cannot be awaited. `httpx` has a nearly identical API plus an
async client:

```python
async with httpx.AsyncClient() as client:
    responses = await asyncio.gather(*(client.get(url) for url in urls))
```

In [ ]:
import asyncio

import httpx


async def fetch_async(concurrency):
    limits = httpx.Limits(max_connections=concurrency)
    async with httpx.AsyncClient(base_url=BASE, timeout=10, limits=limits) as client:
        async def one(job_id):
            response = await client.get(f"/jobs/{job_id}")
            response.raise_for_status()
            payload = response.json()
            return Job(id=payload["id"], state=payload["state"])

        return await asyncio.gather(*(one(j) for j in TARGETS))


started = time.perf_counter()
jobs = asyncio.run(fetch_async(20))
async_elapsed = time.perf_counter() - started

fastest_threads = min(timings[k] for k in timings if k.startswith("threads"))
fastest_label = min((k for k in timings if k.startswith("threads")),
                    key=lambda k: timings[k])

print(f"   {'serial':16} {timings['serial'] * 1000:7.0f} ms")
print(f"   {fastest_label:16} {fastest_threads * 1000:7.0f} ms"
      f"  {baseline / fastest_threads:5.1f}x   <- best thread result")
print(f"   {'asyncio (20)':16} {async_elapsed * 1000:7.0f} ms"
      f"  {baseline / async_elapsed:5.1f}x")
print()
faster = "asyncio slower" if async_elapsed > fastest_threads else "asyncio faster"
print(f"   best threads vs asyncio: {async_elapsed / fastest_threads:.2f}x ({faster})")

🔴 **On this benchmark, threads beat asyncio** — and the *reason* matters far
more than the number.

Three things are true here that are not true of a real API:

1. **The bottleneck is the server, not the client.** `ThreadingHTTPServer` spawns an OS thread
   per connection; it becomes the constraint long before the client does.
2. **There is no network.** Loopback has microsecond latency, so asyncio's core advantage —
   not blocking a thread during a *long* wait — has almost nothing to exploit.
3. **`httpx` does more per request than `requests`**, and the event loop plus `AsyncClient`
   setup is a fixed cost paid on every run.

> **Do not take "threads are faster than asyncio" away from this.** Take away that
> **you must measure your own workload** (**17.5**), because the shape of the bottleneck decides
> the answer.

### When each actually wins

| | **Threads** (**12.4**) | **asyncio** (**12.5**) |
|---|---|---|
| Concurrency ceiling | hundreds — each thread costs ~8 MB of stack | 🔴 **tens of thousands** |
| Works with | `requests`, any blocking library | only async-aware libraries |
| Cost to adopt | 🔴 **almost none** — wrap it in a pool | `async` all the way down (**12.5**) |
| Best at | 🔴 **most API work: tens to hundreds of calls** | many thousands of concurrent connections |
| Debugging | ordinary stack traces (**15.8**) | harder; the debugger sees the loop |

🔴 **For the vast majority of API clients, `ThreadPoolExecutor` is the right answer**, and it is
one line. Reach for asyncio when you genuinely have thousands of concurrent connections, or when
the framework you are in is already async.

## 🔴 Connection pooling: the cheapest win

Before any concurrency at all, there is a change worth measuring — and it is the one **11.5**
introduced with `Session`.

Every new connection costs a TCP handshake (and a TLS handshake, on HTTPS). Reusing one
connection skips both. With the artificial latency switched **off**, that cost is visible.

In [ ]:
DELAY["seconds"] = 0.0          # remove the fake latency so setup cost is visible
COUNT = 120


def without_pooling():
    for i in range(COUNT):
        requests.get(f"{BASE}/jobs/b{i}", timeout=10)     # a NEW connection each time


def with_pooling():
    with requests.Session() as ses:                       # one connection, reused
        for i in range(COUNT):
            ses.get(f"{BASE}/jobs/b{i}", timeout=10)


pooling = {}
for label, call in (("new connection each", without_pooling),
                    ("Session (pooled)", with_pooling)):
    started = time.perf_counter()
    call()
    pooling[label] = time.perf_counter() - started
    print(f"   {label:22} {COUNT} requests in {pooling[label] * 1000:7.0f} ms")

slow, fast = pooling.values()
print()
print(f"   pooling is {slow / fast:.0f}x faster here, on loopback with no TLS.")
print("   🔴 Over a real network with HTTPS the gap is LARGER, because you are")
print("      also skipping a TLS handshake on every request.")

DELAY["seconds"] = 0.15          # restore for the rest of the notebook

A large difference, from **using a `Session`** — no concurrency, no new
libraries, one object.

🔴 **Do this before reaching for threads.** It is the highest ratio of benefit to effort in this
entire folder, and it is the single most common thing missing from a slow API client.

## Bounded concurrency, because rate limits exist

**18.3** covered rate limits. Concurrency and rate limits interact badly: firing 200 requests at
once is the fastest possible way to collect a `429` and get your key throttled.

**Bound it.** `max_workers` does this for threads; a `Semaphore` does it for asyncio.

In [ ]:
async def fetch_bounded(limit):
    """A semaphore caps how many requests are in flight at once."""
    gate = asyncio.Semaphore(limit)
    in_flight = {"now": 0, "peak": 0}

    async with httpx.AsyncClient(base_url=BASE, timeout=10) as client:
        async def one(job_id):
            async with gate:
                in_flight["now"] += 1
                in_flight["peak"] = max(in_flight["peak"], in_flight["now"])
                try:
                    response = await client.get(f"/jobs/{job_id}")
                    return response.status_code
                finally:
                    in_flight["now"] -= 1

        results = await asyncio.gather(*(one(j) for j in TARGETS))
    return results, in_flight["peak"]


for limit in (3, 8):
    started = time.perf_counter()
    results, peak = asyncio.run(fetch_bounded(limit))
    elapsed = time.perf_counter() - started
    print(f"   semaphore({limit}): {len(results)} requests, peak in flight = {peak},"
          f" {elapsed * 1000:6.0f} ms")

print()
print("🔴 The peak never exceeds the limit. That is the point: you choose the")
print("   pressure you put on someone else's service, rather than discovering")
print("   it from a 429 (18.3).")
print()
print("   For threads, max_workers is the same control.")

The peak in-flight count never exceeded the semaphore, and the smaller limit
took longer — which is the trade you are choosing deliberately.

> **Combine this with 18.3's retry policy.** Bounded concurrency reduces how often you hit a
> rate limit; `Retry-After` handles it when you do. Neither replaces the other.

## Timeouts, once more

**11.5** said timeouts are not optional. Under concurrency it is worse: a request with no
timeout does not just hang — it **holds a worker forever**, and a pool of twenty workers can be
fully consumed by twenty hung requests while the rest of your program starves.

```python
session.get(url, timeout=(3.05, 27))      # (connect, read)
httpx.AsyncClient(timeout=httpx.Timeout(10.0, connect=5.0))
```

🔴 A connect timeout and a read timeout are different failures: one means *the server is not
reachable*, the other means *it is reachable and slow* (**18.1**'s exception taxonomy).

## Interview Questions

1. **How would you test a class that calls an HTTP API?** *(fake at your own boundary; a local
   server for integration; do not mock `requests`)*
2. **Why not `patch("requests.get")`?** *(it encodes your beliefs about `requests`; **15.5**)*
3. **What does a recorded fixture buy you, and what does it hide?** *(speed; a frozen API, and
   possibly credentials in the recording)*
4. **Your API client is slow. `cProfile` shows nothing. What is happening?** *(I/O-bound —
   blocked in a socket read, which is not CPU time; **17.5**)*
5. **Threads or asyncio for 50 API calls?** *(threads — one line, works with `requests`, and the
   measurement in this notebook)*
6. **When does asyncio genuinely win?** *(thousands of concurrent connections, or an already-async
   codebase)*
7. **Why do threads help here when **12.1** said the GIL prevents parallelism?** *(a thread
   waiting on a socket has released the GIL; the GIL only serialises bytecode)*
8. **What is the cheapest way to speed up a client making many requests?** *(a `Session` —
   connection reuse, measured above)*
9. **You have 5,000 URLs and a rate limit of 10 requests per second. Design the client.**
   *(bounded concurrency + `Retry-After` + backoff with jitter; **18.3**)*
10. **A request has no timeout and the server hangs. What happens to a thread pool?** *(a worker
    is consumed permanently; enough of them and the pool is dead)*

In [ ]:
# ---- tidy up ----
import shutil

SERVER.shutdown()
shutil.rmtree(WORK, ignore_errors=True)
print("fake API stopped, scratch removed:", not WORK.exists())
print("requests the server handled:", REQUEST_COUNT["n"])

---

## Common Mistakes & Pitfalls

1. 🔴 **Mocking `requests` itself.** You end up testing your beliefs about a library rather than your code (**15.5**).
2. **Having only unit tests.** They cannot catch a wrong URL shape or a real status behaving differently from your fake.
3. **Letting integration tests run in the normal suite.** Mark them and exclude them (**15.3**) — a pre-commit hook should not need a server.
4. 🔴 **Committing a recorded fixture without reading it.** They routinely capture credentials (**18.2**) and freeze the API as it was.
5. **CPU-profiling an I/O-bound client.** The profile is empty and every micro-optimisation is invisible (**17.5**).
6. 🔴 **Not using a `Session`.** A new connection per request is the most common cause of a slow client, and the easiest to fix.
7. **Assuming asyncio is faster.** Measure it — on this workload threads won, and the reason is the shape of the bottleneck.
8. **Unbounded concurrency.** Two hundred simultaneous requests is the fastest way to earn a `429` (**18.3**).
9. **Omitting a timeout under concurrency.** One hung request holds a worker forever; enough of them kill the pool.
10. **Sharing one `Session` across processes.** It is thread-safe enough for a pool, but not something to fork with (**12.3**).

## Best Practices

- Wrap the API in a client class that accepts its `Session` — testable without patching.
- Unit-test against a fake at your own boundary; integration-test against a local server.
- Mark integration tests and keep them out of the fast suite.
- 🔴 Use a `Session` before doing anything else about performance.
- Reach for `ThreadPoolExecutor` first for concurrent HTTP; it works with `requests` and costs one line.
- Use asyncio and `httpx` when you have thousands of concurrent connections, or are already async.
- Bound concurrency deliberately — `max_workers` or a `Semaphore` — and pair it with a retry policy (**18.3**).
- Always set both a connect and a read timeout.
- Measure before and after (**17.5**); the bottleneck is rarely where you assumed.

## Practice Exercises

Try these before moving on.

1. Add a `list_jobs` method to `JobsClient` and write both a unit test with `FakeSession` and an integration test against the fixture server.
2. 🔴 Rewrite `test_parses_a_job` using `patch("requests.get")` instead. Which version would survive switching the client to `httpx`?
3. Add a test asserting the client retries a `503` (**18.3**) but not a `404`. Which layer does that test belong in?
4. Run the thread benchmark with 1, 2, 5, 10, 20 and 50 workers. Where do the returns stop, and why is it near the number of requests?
5. 🔴 Re-run the threads-vs-asyncio comparison with the server's `DELAY` set to 1.0 s and 200 targets. Does the answer change? Explain why.
6. Measure `Session` vs no `Session` against an HTTPS URL. Is the gap bigger than on loopback? Why?
7. Implement a rate-limited client: at most 10 requests per second, using a semaphore and a timestamp queue. Verify it never exceeds the rate.
8. Remove the timeout from `JobsClient`, point it at the `/slow` endpoint, and watch a thread pool of 5 stop responding.
9. **Interview question:** you have 5,000 URLs, a 10 req/s limit and a 2% failure rate. Sketch the client — concurrency, retries, and how long it takes.

---

## Version notes

| Version | Change |
|---|---|
| **httpx 0.28** | `AsyncClient` with `httpx.Limits`; a `requests`-compatible sync API and HTTP/2 support |
| **Python 3.11** | `asyncio.TaskGroup` — a better `gather` with proper cancellation (**12.5**) |
| **Python 3.9** | `asyncio.to_thread()` — run one blocking call from async code without a pool |
| **requests 2.x** | `Session` pools connections per host via `urllib3`; tune with `HTTPAdapter(pool_maxsize=...)` |

> **`asyncio.to_thread`** is the bridge worth knowing: in an async program that must call one
> blocking library, `await asyncio.to_thread(blocking_call, arg)` avoids rewriting everything.

## 18 Working with APIs — the folder

| Notebook | Covers |
|---|---|
| **18.1** | REST semantics, status codes, JSON in and out, the exception taxonomy |
| **18.2** | authentication schemes, tokens, secrets, redaction, a real leak case study |
| **18.3** | pagination as a generator, rate limits, backoff with jitter, idempotency keys |
| **18.4** | validating untrusted responses with `pydantic` |
| **18.5** | this notebook — testing API clients, and concurrency for I/O-bound work |

**The one-sentence version:** *treat the API as untrusted and unreliable — validate what it
sends, retry what deserves retrying, never log the credential, and put a boundary around it so
you can test and replace it.*

## Related

- **15.5 Test Doubles** — "wrap what you do not own", which this notebook applies to HTTP
- **15.4 Fixtures** — the session-scoped server fixture
- **12.4 concurrent.futures** — `ThreadPoolExecutor`, the right default here
- **12.5 asyncio** — the async model `httpx` plugs into
- **17.5 Profiling** — why an I/O-bound profile is empty
- **11.5 HTTP** — `Session`, connection reuse and timeouts
- **19 Capstone Projects** — where a tested, resilient client becomes part of something larger